<a href="https://colab.research.google.com/github/adhiyaaa23/Kasir-Chatbot/blob/main/Chatbot%20Kasir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Setup (Jalankan Cell Ini Terlebih Dahulu)

1. Buka **Secrets** (kunci icon di sidebar kiri)
2. Klik **"Create new secret"**
3. Isi:
   - Name: `GEMINI_API_KEY`
   - Value: Paste API key dari https://aistudio.google.com/app/apikey
4. Jalankan cell di bawah

In [ ]:
# Setup: Install dependencies
!pip install -q google-genai

In [ ]:
import os
from google import genai
from google.genai import types

# 1. Definisikan fungsi tool
def get_menu(menu_type: str) -> dict:
    """Ambil menu berdasarkan tipe (food atau drink)"""
    if menu_type == "food":
        return {
            "nasi goreng": 15000,
            "mie goreng": 12000,
            "sate ayam": 20000
        }
    elif menu_type == "drink":
        return {
            "es teh": 5000,
            "es jeruk": 7000,
            "kopi": 10000
        }
    else:
        return {"error": "Menu type not found"}

# 2. Inisialisasi client Gemini dengan environment variable
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise ValueError("GEMINI_API_KEY tidak ditemukan. Tambahkan di Secrets terlebih dahulu!")

client = genai.Client(api_key=api_key)

# 3. System instruction
system_instruction = """
Anda adalah Budi, seorang pelayan restoran bintang lima yang sangat ramah, hangat, dan cekatan.
Tugas utama Anda adalah melayani pelanggan dan menghitung total tagihan mereka.

Aturan Penting:
1. Anda TIDAK BOLEH menebak harga. Anda WAJIB memanggil fungsi get_menu untuk mencari harga dari item yang dipesan.
2. Panggil fungsi dengan parameter \"food\" untuk makanan, dan \"drink\" untuk minuman.
3. STRUKTUR JAWABAN WAJIB (Harus berurutan seperti di bawah ini):
   - PERTAMA: Ulangi daftar pesanan pelanggan secara terperinci beserta harga masing-masing item.
   - KEDUA: Hitung dan tampilkan total harga keseluruhan pesanan dengan akurat.
   - KETIGA: Setelah total disebutkan, tanyakan dengan sopan apakah ada tambahan pesanan atau hal lain yang bisa dibantu.
4. Gunakan bahasa yang sopan dan gaya bahasa layaknya pelayan profesional.
"""

# 4. Konfigurasi model
config = types.GenerateContentConfig(
    tools=[get_menu],
    system_instruction=system_instruction,
    temperature=0.1,
)

# 5. Buat sesi chat
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=config
)

# 6. Bagian Interaktif
print("=== Simulasi Restoran Dimulai ===")
print("Silakan pesan makanan Anda. (Ketik 'keluar' untuk menghentikan program)\n")

# Memancing percakapan pertama
pertanyaan_awal = "Saya pesan Sate ayam, Nasi goreng, dan es teh. Berapa totalnya?"
print(f"Anda: {pertanyaan_awal}")
response = chat.send_message(pertanyaan_awal)
print(f"Budi (Gemini):\n{response.text}\n")
print("-" * 40)

# Membuka kolom chat untuk user membalas
while True:
    user_input = input("Anda: ")

    # Deteksi jika user ingin menyudahi program
    if user_input.lower() in ['keluar', 'tidak', 'cukup', 'sudah']:
        print("Budi (Gemini): Baik, terima kasih banyak atas pesanan Anda. Mohon ditunggu sebentar, pesanan akan segera kami siapkan. Selamat menikmati!")
        break

    # Mengirim balasan user ke Gemini
    response = chat.send_message(user_input)

    # Menampilkan balasan Budi
    print(f"Budi (Gemini):\n{response.text}\n")
    print("-" * 40)